# Lab 7: Model Deployment for Medical Image Segmentation

## Overview
Deploy a trained MONAI segmentation model (SegResNet or SwinUNETR) to a SageMaker async endpoint for production inference on 3D medical images (NIfTI format).

## Learning Objectives
- Package a trained segmentation model for SageMaker deployment
- Deploy to an async endpoint (ideal for large 3D volumes)
- Configure auto-scaling with scale-to-zero
- Invoke the endpoint with NIfTI files via S3
- Process and visualize segmentation results

## Prerequisites
- Completed Lab 1 or Lab 2 (trained model artifacts in S3)
- IAM role with SageMaker and S3 permissions

## Why Async Endpoints?
Medical image segmentation on 3D volumes (CT/MRI) can take 30-120 seconds per scan. Async endpoints are ideal because:
- No timeout limits (real-time endpoints timeout at 60s)
- Scale-to-zero when idle (cost savings)
- Queue-based processing for batch workloads
- Results delivered to S3

**Estimated Time:** 30-45 minutes

## Step 1: Setup Environment

In [1]:
%pip install sagemaker boto3 nibabel numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sagemaker
import boto3
import json
import time
import base64
import tempfile
import numpy as np
from sagemaker.pytorch import PyTorchModel
from sagemaker.async_inference import AsyncInferenceConfig
from sagemaker import get_execution_role

import sys
sys.path.append("../code/scripts")
from get_or_create_role import get_or_create_sagemaker_role

role = get_or_create_sagemaker_role()


sagemaker_session = sagemaker.Session(boto3.Session(region_name='us-east-1'))
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()

print(f"Region: {region}")
print(f"Bucket: {bucket}")
print(f"Role: {role}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ubuntu/.config/sagemaker/config.yaml
Role already exists: arn:aws:iam::575108919340:role/AmazonSageMaker-ExecutionRole-sgm

Using role: arn:aws:iam::575108919340:role/AmazonSageMaker-ExecutionRole-sgm
Region: us-east-1
Bucket: sagemaker-us-east-1-575108919340
Role: arn:aws:iam::575108919340:role/AmazonSageMaker-ExecutionRole-sgm


## Step 2: Configure Model Artifacts

Point to the trained model artifacts from a previous training job. The `model.tar.gz` should contain:
- `best_model.pth` or `final_model.pth` (model weights)
- `config.json` (optional, specifies model architecture)

If your training job used Lab 1 or Lab 2, the model artifacts are stored in the S3 output path.

In [2]:
# Replace with your training job's model artifact S3 path
# model_data = "s3://YOUR_BUCKET/segmentation_data/output/YOUR_TRAINING_JOB/output/model.tar.gz"
model_data = "s3://public-datasets-imaging-us-east-1/segmentation_data/output/medical-seg-simple-2026-06-12-15-47-25-720/output/model.tar.gz"

# PyTorch inference container
image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.1.0",
    py_version="py310",
    instance_type="ml.g5.xlarge",
    image_scope="inference"
)

print(f"Model data: {model_data}")
print(f"Image URI: {image_uri}")

Model data: s3://public-datasets-imaging-us-east-1/segmentation_data/output/medical-seg-simple-2026-06-12-15-47-25-720/output/model.tar.gz
Image URI: 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.1.0-gpu-py310


## Step 3: Configure Async Endpoint

Async endpoints store input and output in S3, enabling processing of large 3D volumes without timeout constraints.

In [3]:
endpoint_name = "medical-segmentation-async-endpoint"

# S3 paths for async I/O
async_output_path = f"s3://{bucket}/segmentation-inference/output"
async_failure_path = f"s3://{bucket}/segmentation-inference/failures"

async_config = AsyncInferenceConfig(
    output_path=async_output_path,
    failure_path=async_failure_path,
    max_concurrent_invocations_per_instance=2  # 3D segmentation is memory-intensive
)

print(f"Endpoint name: {endpoint_name}")
print(f"Output path: {async_output_path}")
print(f"Failure path: {async_failure_path}")

Endpoint name: medical-segmentation-async-endpoint
Output path: s3://sagemaker-us-east-1-575108919340/segmentation-inference/output
Failure path: s3://sagemaker-us-east-1-575108919340/segmentation-inference/failures


## Step 4: Deploy the Model

Create a PyTorchModel pointing to the inference script in `./deploy/` and deploy to an async endpoint.

**Instance Choice:**
- `ml.g5.xlarge`: 1 GPU (24GB A10G) — good for SegResNet
- `ml.g5.2xlarge`: 1 GPU (24GB A10G) + more CPU/RAM — for larger volumes
- `ml.g4dn.xlarge`: 1 GPU (16GB T4) — budget option

In [4]:
sm_client = boto3.client("sagemaker")
existing_endpoints = sm_client.list_endpoints()["Endpoints"]

if any(ep["EndpointName"] == endpoint_name for ep in existing_endpoints):
    print(f"Endpoint '{endpoint_name}' already exists. Skipping deployment.")
else:
    pytorch_model = PyTorchModel(
        model_data=model_data,
        role=role,
        source_dir="deploy",
        entry_point="inference.py",
        framework_version="2.1.0",
        py_version="py310",
        image_uri=image_uri,
        sagemaker_session=sagemaker_session,
        env={
            "SAGEMAKER_MODEL_SERVER_TIMEOUT": "300",  # 5 min timeout for large volumes
            "SAGEMAKER_MODEL_SERVER_WORKERS": "1",    # 1 worker per GPU
        }
    )

    predictor = pytorch_model.deploy(
        instance_type="ml.g5.xlarge",
        initial_instance_count=1,
        endpoint_name=endpoint_name,
        async_inference_config=async_config,
    )
    print(f"Endpoint '{endpoint_name}' deployed successfully.")

---------!Endpoint 'medical-segmentation-async-endpoint' deployed successfully.


## Step 5: Configure Auto-Scaling

Enable scale-to-zero when idle and scale-out under load. This is critical for cost management since segmentation endpoints are typically used in bursts (e.g., after a batch of scans arrives).

In [5]:
aas_client = boto3.client("application-autoscaling")
resource_id = f"endpoint/{endpoint_name}/variant/AllTraffic"

# Register scalable target: scale between 0 and 3 instances
aas_client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=0,
    MaxCapacity=3,
)
print("Registered scalable target: min=0, max=3")

# Scale based on queue backlog per instance
aas_client.put_scaling_policy(
    PolicyName=f"{endpoint_name}-scaling-policy",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 3.0,  # Scale out when backlog > 3 per instance
        "CustomizedMetricSpecification": {
            "MetricName": "ApproximateBacklogSizePerInstance",
            "Namespace": "AWS/SageMaker",
            "Dimensions": [{"Name": "EndpointName", "Value": endpoint_name}],
            "Statistic": "Average",
        },
        "ScaleInCooldown": 600,   # Wait 10 min before scaling in
        "ScaleOutCooldown": 60,   # Scale out quickly
    },
)
print("Applied scaling policy: target backlog=3, scale-in cooldown=600s, scale-out cooldown=60s")

Registered scalable target: min=0, max=3
Applied scaling policy: target backlog=3, scale-in cooldown=600s, scale-out cooldown=60s


## Step 6: Invoke the Endpoint

Submit a NIfTI scan for segmentation. The input is uploaded to S3 and the endpoint processes it asynchronously.

In [6]:
class SegmentationAsyncPredictor:
    """Helper class to invoke async segmentation endpoint and retrieve results."""

    def __init__(self, endpoint_name, bucket, region="us-east-1"):
        self.endpoint_name = endpoint_name
        self.bucket = bucket
        self.runtime_client = boto3.client("sagemaker-runtime", region_name=region)
        self.s3_client = boto3.client("s3", region_name=region)

    def predict_from_s3(self, s3_nifti_uri):
        """Submit an S3-hosted NIfTI file for segmentation."""
        payload = json.dumps({"file_path": s3_nifti_uri})
        input_key = f"segmentation-inference/input/request_{int(time.time())}.json"
        self.s3_client.put_object(Bucket=self.bucket, Key=input_key, Body=payload)
        input_location = f"s3://{self.bucket}/{input_key}"

        response = self.runtime_client.invoke_endpoint_async(
            EndpointName=self.endpoint_name,
            InputLocation=input_location,
            ContentType="application/json",
        )
        output_location = response["OutputLocation"]
        print(f"Request submitted. Output: {output_location}")
        return output_location

    def predict_from_local(self, local_nifti_path):
        """Upload a local NIfTI file to S3 and submit for segmentation."""
        filename = os.path.basename(local_nifti_path)
        s3_key = f"segmentation-inference/input/{filename}"
        self.s3_client.upload_file(local_nifti_path, self.bucket, s3_key)
        s3_uri = f"s3://{self.bucket}/{s3_key}"
        print(f"Uploaded to: {s3_uri}")
        return self.predict_from_s3(s3_uri)

    def get_result(self, output_location, timeout=600, poll_interval=10):
        """Wait for and retrieve the segmentation result."""
        parts = output_location.replace("s3://", "").split("/", 1)
        out_bucket, out_key = parts[0], parts[1]

        start_time = time.time()
        print(f"Waiting for result...")

        while time.time() - start_time < timeout:
            try:
                obj = self.s3_client.get_object(Bucket=out_bucket, Key=out_key)
                result = json.loads(obj["Body"].read().decode("utf-8"))
                elapsed = int(time.time() - start_time)
                print(f"Result received in {elapsed}s")
                return result
            except self.s3_client.exceptions.NoSuchKey:
                elapsed = int(time.time() - start_time)
                print(f"  Waiting... ({elapsed}s)", end="\r")
                time.sleep(poll_interval)

        raise TimeoutError(f"No result after {timeout}s")

    @staticmethod
    def decode_segmentation(result, output_path=None):
        """Decode base64 segmentation mask and optionally save to disk."""
        import nibabel as nib

        seg_bytes = base64.b64decode(result["segmentation_mask_base64"])

        if output_path is None:
            output_path = tempfile.mktemp(suffix="_seg.nii.gz")

        with open(output_path, "wb") as f:
            f.write(seg_bytes)

        seg_img = nib.load(output_path)
        print(f"Segmentation saved to: {output_path}")
        print(f"Shape: {seg_img.shape}")
        return seg_img, output_path


import os
predictor = SegmentationAsyncPredictor(endpoint_name, bucket, region)

## Check Endpoint

Check if the endpoint is warm or cold. 

In [ ]:
sm_client = boto3.client("sagemaker")
desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
instance_count = desc['ProductionVariants'][0]['CurrentInstanceCount']

if instance_count > 0:
    print(f"Endpoint is WARM — {instance_count} instance(s) running")
else:
    print("Endpoint is COLD — scaled to zero, next request will trigger cold start (~8-12 min)")



Endpoint is COLD — scaled to zero, next request will trigger cold start (~8-12 min)


#### Manually Invoking the endpoint to a warmstate

Option 1: Set `MinCapacity` to 1 (keep it always warm)

```
aas_client = boto3.client("application-autoscaling")
resource_id = f"endpoint/{endpoint_name}/variant/AllTraffic"

aas_client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=1,  # Never scale to zero
    MaxCapacity=3,
)
```
This guarantees at least one instance is always running. Costs ~$1.41/hr for ml.g5.xlarge even when idle, but you'll never hit a cold start.

Option 2: Manually update the instance count (warm it on-demand)

```
sm_client = boto3.client("sagemaker")

sm_client.update_endpoint_weights_and_capacities(
    EndpointName=endpoint_name,
    DesiredWeightsAndCapacities=[{
        "VariantName": "AllTraffic",
        "DesiredInstanceCount": 1,
    }]
)
print("Scaling up... wait ~8-12 min for instance to be ready")
```



### Lets warm it up

In [17]:
sm_client = boto3.client("sagemaker")

sm_client.update_endpoint_weights_and_capacities(
    EndpointName=endpoint_name,
    DesiredWeightsAndCapacities=[{
        "VariantName": "AllTraffic",
        "DesiredInstanceCount": 1,
    }]
)
print("Scaling up... wait ~8-12 min for instance to be ready")

ClientError: An error occurred (ValidationException) when calling the UpdateEndpointWeightsAndCapacities operation: Cannot update in-progress endpoint "medical-segmentation-async-endpoint".

In [18]:
# Option A: Submit from S3 URI directly
# s3_scan_uri = "s3://YOUR_BUCKET/segmentation_data/test/subject_01/img.nii.gz"  # Replace

s3_scan_uri="s3://public-datasets-imaging-us-east-1/segmentation_data/test/10090344/img.nii.gz"

output_location = predictor.predict_from_s3(s3_scan_uri)
result = predictor.get_result(output_location)

# Display results summary
print(f"\n{'='*50}")
print(f"Model: {result['model_name']}")
print(f"Original volume shape: {result['original_shape']}")
print(f"Segmentation shape: {result['segmentation_shape']}")
print(f"Segmented voxels: {result['segmented_voxels']:,}")
print(f"Total voxels: {result['total_voxels']:,}")
print(f"Segmentation coverage: {result['segmentation_percentage']:.2f}%")
print(f"{'='*50}")

Request submitted. Output: s3://sagemaker-us-east-1-575108919340/segmentation-inference/output/5b11d5ec-c155-48b7-93c7-7e2bef8ba2f1.out
Waiting for result...
Result received in 10s

Model: SegResNet
Original volume shape: [512, 512, 275]
Segmentation shape: [128, 128, 64]
Segmented voxels: 184,387
Total voxels: 1,048,576
Segmentation coverage: 17.58%


In [ ]:
# Option B: Submit from local file
# local_scan = "/path/to/your/scan.nii.gz"
# output_location = predictor.predict_from_local(local_scan)
# result = predictor.get_result(output_location)

## Step 7: Decode and Visualize Segmentation

In [12]:
# Decode the segmentation mask
seg_img, seg_path = SegmentationAsyncPredictor.decode_segmentation(
    result, output_path="./segmentation_output.nii.gz"
)

# View a slice
seg_data = seg_img.get_fdata()
print(f"\nSegmentation statistics:")
print(f"  Unique values: {np.unique(seg_data)}")
print(f"  Non-zero voxels: {np.count_nonzero(seg_data):,}")
print(f"  Volume shape: {seg_data.shape}")

Segmentation saved to: ./segmentation_output.nii.gz
Shape: (128, 128, 64)

Segmentation statistics:
  Unique values: [0. 1.]
  Non-zero voxels: 184,387
  Volume shape: (128, 128, 64)


In [11]:
# Optional: Visualize middle slices
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    mid_x = seg_data.shape[0] // 2
    mid_y = seg_data.shape[1] // 2
    mid_z = seg_data.shape[2] // 2

    axes[0].imshow(seg_data[mid_x, :, :], cmap="hot")
    axes[0].set_title(f"Sagittal (x={mid_x})")
    axes[0].axis("off")

    axes[1].imshow(seg_data[:, mid_y, :], cmap="hot")
    axes[1].set_title(f"Coronal (y={mid_y})")
    axes[1].axis("off")

    axes[2].imshow(seg_data[:, :, mid_z], cmap="hot")
    axes[2].set_title(f"Axial (z={mid_z})")
    axes[2].axis("off")

    plt.suptitle("Segmentation Mask - Middle Slices")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping visualization")

matplotlib not installed, skipping visualization


## Step 8: Batch Inference

Submit multiple scans for batch processing. The async endpoint queues them automatically.

In [19]:
# Batch inference example
scan_uris = [
    # "s3://YOUR_BUCKET/segmentation_data/test/subject_01/img.nii.gz",
    # "s3://YOUR_BUCKET/segmentation_data/test/subject_02/img.nii.gz",
    # "s3://YOUR_BUCKET/segmentation_data/test/subject_03/img.nii.gz",
]
scan_uris = ["s3://public-datasets-imaging-us-east-1/segmentation_data/test/10090344/img.nii.gz",
"s3://public-datasets-imaging-us-east-1/segmentation_data/test/10095371/img.nii.gz"]

# Submit all scans
output_locations = []
for uri in scan_uris:
    output_loc = predictor.predict_from_s3(uri)
    output_locations.append(output_loc)
    time.sleep(1)  # Small delay between submissions

print(f"\nSubmitted {len(output_locations)} scans for processing")

# Collect results
results = []
for i, output_loc in enumerate(output_locations):
    print(f"\nRetrieving result {i+1}/{len(output_locations)}...")
    result = predictor.get_result(output_loc)
    results.append(result)
    print(f"  Segmentation coverage: {result['segmentation_percentage']:.2f}%")

print(f"\nAll {len(results)} results collected.")

Request submitted. Output: s3://sagemaker-us-east-1-575108919340/segmentation-inference/output/ad136e71-e7f6-4e90-a99b-8876756f4feb.out
Request submitted. Output: s3://sagemaker-us-east-1-575108919340/segmentation-inference/output/e8a931ac-063f-449b-aacd-d59692780879.out

Submitted 2 scans for processing

Retrieving result 1/2...
Waiting for result...
Result received in 10s
  Segmentation coverage: 17.58%

Retrieving result 2/2...
Waiting for result...
Result received in 10s
  Segmentation coverage: 17.76%

All 2 results collected.


## Step 9: Monitor Endpoint

In [20]:
# Check endpoint status
sm_client = boto3.client("sagemaker")
desc = sm_client.describe_endpoint(EndpointName=endpoint_name)

print(f"Endpoint: {endpoint_name}")
print(f"Status: {desc['EndpointStatus']}")
print(f"Instance type: {desc['ProductionVariants'][0].get('CurrentInstanceCount', 'N/A')} instance(s)")
print(f"Created: {desc['CreationTime']}")
print(f"Last modified: {desc['LastModifiedTime']}")

Endpoint: medical-segmentation-async-endpoint
Status: InService
Instance type: 2 instance(s)
Created: 2026-06-12 16:04:06.809000+00:00
Last modified: 2026-06-12 17:11:32.401000+00:00


## Step 10: Cleanup

Delete the endpoint, endpoint configuration, and model to stop incurring costs.

In [22]:
# Uncomment to delete resources
# WARNING: This will delete the endpoint and all associated resources

cleanup = True  # Set to True to delete

if cleanup:
    sm_client = boto3.client("sagemaker")
    aas_client = boto3.client("application-autoscaling")

    # Deregister auto-scaling
    try:
        resource_id = f"endpoint/{endpoint_name}/variant/AllTraffic"
        aas_client.deregister_scalable_target(
            ServiceNamespace="sagemaker",
            ResourceId=resource_id,
            ScalableDimension="sagemaker:variant:DesiredInstanceCount",
        )
        print(f"Deregistered auto-scaling for: {endpoint_name}")
    except Exception as e:
        print(f"Auto-scaling cleanup: {e}")

    # Delete endpoint, config, and model
    try:
        desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
        endpoint_config_name = desc["EndpointConfigName"]
        config_desc = sm_client.describe_endpoint_config(
            EndpointConfigName=endpoint_config_name
        )
        model_name = config_desc["ProductionVariants"][0]["ModelName"]

        sm_client.delete_endpoint(EndpointName=endpoint_name)
        print(f"Deleted endpoint: {endpoint_name}")

        sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
        print(f"Deleted endpoint config: {endpoint_config_name}")

        sm_client.delete_model(ModelName=model_name)
        print(f"Deleted model: {model_name}")
    except Exception as e:
        print(f"Cleanup error: {e}")
else:
    print("Cleanup skipped. Set cleanup=True to delete resources.")

Deregistered auto-scaling for: medical-segmentation-async-endpoint
Deleted endpoint: medical-segmentation-async-endpoint
Deleted endpoint config: medical-segmentation-async-endpoint
Deleted model: pytorch-inference-2026-06-12-16-04-05-362


## Deployment Architecture

```
Client (NIfTI scan)
    │
    ▼
S3 Input Bucket ──► Async Endpoint (ml.g5.xlarge)
                         │
                         ├── model_fn: Load SegResNet/SwinUNETR
                         ├── input_fn: Parse S3/local/base64 NIfTI
                         ├── predict_fn: Sliding window inference
                         └── output_fn: JSON + base64 segmentation
                         │
                         ▼
                    S3 Output Bucket (JSON result)
```

## Cost Estimates

| Instance | GPU | Cost/Hour | Per Scan (~60s) |
|----------|-----|-----------|----------------|
| ml.g4dn.xlarge | T4 16GB | $0.94 | ~$0.016 |
| ml.g5.xlarge | A10G 24GB | $1.41 | ~$0.024 |
| ml.g5.2xlarge | A10G 24GB | $1.89 | ~$0.032 |

With scale-to-zero, you only pay when the endpoint is actively processing scans.

## Key Takeaways

- **Async endpoints** are ideal for 3D medical image segmentation (no timeout limits)
- **Scale-to-zero** eliminates costs when the endpoint is idle
- **Sliding window inference** enables processing of arbitrarily large volumes
- **Base64 encoding** of results allows returning segmentation masks in JSON responses
- For production, consider adding SNS notifications for completed inferences

## Next Steps

- Add CloudWatch alarms for endpoint health monitoring
- Integrate with Step Functions for automated pipelines
- Set up SNS notifications when inference completes
- Use SageMaker Batch Transform for large-scale offline processing